In [ ]:
import cv2
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import math

## to be removed before final 


In [48]:
def inspect_mask(mask, CLASS_NAMES):
    """
    Prints class IDs, class names, and pixel counts.
    """

    if mask is None:
        raise ValueError("Mask is None")

    print("Mask shape:", mask.shape)
    print("Mask data type:", mask.dtype)

    unique_values, counts = np.unique(mask, return_counts=True)

    print("\nClasses found in mask:")

    for value, count in zip(unique_values, counts):
        class_name = CLASS_NAMES.get(int(value), "unknown")
        print(f"Class ID: {int(value)} | Name: {class_name} | Pixels: {count}")


def print_mask_rows_columns(mask, start_row=0, end_row=10, start_col=0, end_col=10):
    """
    Prints selected rows and columns from the mask.

    Example:
        start_row=0, end_row=10
        start_col=0, end_col=10

    This prints a 10x10 part of the mask.
    """

    if mask is None:
        raise ValueError("Mask is None")

    print(f"\nPrinting mask rows {start_row} to {end_row - 1}")
    print(f"Printing mask columns {start_col} to {end_col - 1}")

    selected_area = mask[start_row:end_row, start_col:end_col]

    print(selected_area)

    return selected_area
def read_one_mask(mask_path):
    """
    Reads one saved mask image from path.
    This is only for testing.
    """

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Mask not found at path: {mask_path}")

    return mask

In [37]:
inspect_mask(mask, CLASS_NAMES)

print_mask_rows_columns(
    mask,
    start_row=0,
    end_row=10,
    start_col=0,
    end_col=10
    
)

Mask shape: (432, 432)
Mask data type: uint8

Classes found in mask:
Class ID: 0 | Name: background | Pixels: 108776
Class ID: 1 | Name: Human | Pixels: 216
Class ID: 2 | Name: Obstacle | Pixels: 200
Class ID: 3 | Name: Road | Pixels: 53663
Class ID: 4 | Name: Sidewalk | Pixels: 23769

Printing mask rows 0 to 9
Printing mask columns 0 to 9
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]


array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=uint8)

## main code 

In [ ]:
def validate_mask(mask):
    if mask is None:
        raise ValueError("Mask is None")
def get_class_pixels(mask, class_id):
    """
    Returns x and y coordinates of all pixels that belong to one class.
    """
    y_coords, x_coords = np.where(mask == class_id)    
    return x_coords, y_coords
def class_exists(x_coords):
    return len(x_coords) > 0
def calculate_pixel_distances(x_coords, y_coords, bottom_center):
    """
    Calculates distance from bottom-center point to all pixels of one class.
    """
    x_center, y_bottom = bottom_center
    distances = np.sqrt(
        (x_coords - x_center) ** 2 + (y_coords - y_bottom) ** 2
    )
    return distances
def get_nearest_pixel(x_coords, y_coords, distances):
    """
    Finds the nearest pixel and its distance.
    """
    nearest_index = np.argmin(distances)
    nearest_distance = float(distances[nearest_index])
    nearest_x = int(x_coords[nearest_index])
    nearest_y = int(y_coords[nearest_index])
    return nearest_distance, (nearest_x, nearest_y)
def get_side(nearest_x, x_center, center_tolerance=10):
    """
    Decides if nearest point is left, right, or center.
    """
    if nearest_x < x_center - center_tolerance:
        return "left"
    if nearest_x > x_center + center_tolerance:
        return "right"
    return "center"
def build_not_found_result():
    return {
        "distance": float("inf"),
        "nearest_point": None,
        "side": "not_found",
    }
def build_found_result(distance, nearest_point, side):
    return {
        "distance": distance,
        "nearest_point": nearest_point,
        "side": side,
    }
def get_bottom_center_point(frame_or_mask):
    """Returns the (x, y) coordinates of the bottom-center point of the frame or mask."""
    if frame_or_mask is None:
        raise ValueError("Frame or mask is None")
    h, w = frame_or_mask.shape[:2]
    x = w // 2
    y = h - 1
    return x, y
def process_one_class(mask, class_id, class_name, bottom_center, center_tolerance=10):
    """
    Processes only one class.
    Example: Human or Obstacle or Road.
    """
    x_coords, y_coords = get_class_pixels(mask, class_id)   #Give me all row and column positions where mask value equals class_id.
    if not class_exists(x_coords):
        return build_not_found_result()

    distances = calculate_pixel_distances(
        x_coords=x_coords,
        y_coords=y_coords,
        bottom_center=bottom_center
    )

    nearest_distance, nearest_point = get_nearest_pixel(
        x_coords=x_coords,
        y_coords=y_coords,
        distances=distances
    )
    x_center, _ = bottom_center
    nearest_x, _ = nearest_point
    side = get_side(
        nearest_x=nearest_x,
        x_center=x_center,
        center_tolerance=center_tolerance
    )

    return build_found_result(
        distance=nearest_distance,
        nearest_point=nearest_point,
        side=side
    )
def calculate_distance_and_side_from_bottom_center(
    mask,
    bottom_center,
    class_names,
    center_tolerance=10,
):
    """
    Calculates nearest distance and side for all classes.
    """

    validate_mask(mask)

    result = {}

    with ThreadPoolExecutor() as executor:
        outputs = executor.map(
            lambda item: (
                item[1],
                process_one_class(
                    mask=mask,
                    class_id=item[0],
                    class_name=item[1],
                    bottom_center=bottom_center,
                    center_tolerance=center_tolerance
                )
            ),
            class_names.items()
        )
    for class_name, output in outputs:
        result[class_name] = output
    return result
def get_class_info(distance_info, class_name):
    """
    Safely gets one class information from distance_info.
    """

    return distance_info.get(
        class_name,
        {
            "distance": float("inf"),
            "nearest_point": None,
            "side": "not_found",
        },
    )
def is_found(class_info):
    """
    Checks if class exists in the mask.
    """
    return class_info["distance"] != float("inf")
def get_opposite_direction(side):
    """
    If object is on right, turn left.
    If object is on left, turn right.
    If object is center, stay center/stop depending on rule.
    """
    if side == "right":
        return "left"

    if side == "left":
        return "right"

    return "center"

def slow_down(current_velocity, slow_factor=0.4):
    """
    Reduces current velocity.
    """
    return current_velocity * slow_factor
def stop_robot():
    """
    Returns stop command.
    """
    return 0.0

def compute_velocity_from_distances(
    CLASS_NAMES,
    distance_info,
    current_velocity,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4
):
    """
    Converts distance information into simple velocity command.
    """
    class_names = [
    CLASS_NAMES[1],  # Human
    CLASS_NAMES[2],  # Obstacle
    CLASS_NAMES[5],  # Speed Breaker
    CLASS_NAMES[4],  # Sidewalk
    CLASS_NAMES[3],  # Road
]
    def get_distance(class_name):
        return distance_info.get(class_name, {}).get("distance", float("inf"))

    def get_side(class_name):
        return distance_info.get(class_name, {}).get("side", "not_found")

    def opposite_direction(side):
        if side == "right":
            return "left"
        if side == "left":
            return "right"
        return "center"

    def command(velocity, direction, emergency_stop, reason):
        return {
            "linear_velocity": velocity,
            "direction": direction,
            "emergency_stop": emergency_stop,
            "reason": reason,
        }

    # Get distances
    with ThreadPoolExecutor() as executor:
        distance_values = list(executor.map(get_distance, class_names))
    human_dist, obstacle_dist, speed_breaker_dist, sidewalk_dist, road_dist = distance_values
    # Get sides
    with ThreadPoolExecutor() as executor:
        side_values = list(executor.map(get_side, class_names))

    human_side, obstacle_side, speed_breaker_side, sidewalk_side, road_side = side_values

    # 1. Road missing / too far
    if road_dist > road_max_distance:
        return command(0.0, "center", True, "Road missing or too far")

    # 2. Emergency stop for human / obstacle
    if human_dist <= emergency_distance:
        return command(0.0, "center", True, "Human too close")

    if obstacle_dist <= emergency_distance:
        return command(0.0, "center", True, "Obstacle too close")

    # 3. Slow + avoid human / obstacle
    if human_dist <= warning_distance:
        return command(
            current_velocity * slow_factor,
            opposite_direction(human_side),
            False,
            "Human nearby - slow and avoid",
        )
    if obstacle_dist <= warning_distance:
        return command(
            current_velocity * slow_factor,
            opposite_direction(obstacle_side),
            False,
            "Obstacle nearby  slow and avoid",
        )

    # 4. Sidewalk nearby
    if sidewalk_dist <= sidewalk_distance:
        return command(
            current_velocity * slow_factor,
            opposite_direction(sidewalk_side),
            False,
            "Sidewalk nearby - slow and steer away",
        )

    # 5. Speed breaker nearby
    if speed_breaker_dist <= speed_breaker_distance:
        return command(
            current_velocity * slow_factor,
            "center",
            False,
            "Speed breaker nearby - slow down",
        )

    # 6. Safe path
    return command(
        current_velocity,
        "center",
        False,
        "Path is safe",
    )

In [55]:
CLASS_NAMES = {
    0: "background",
    1: "Human",
    2: "Obstacle",
    3: "Road",
    4: "Sidewalk",
    5: "Speed Breaker",
    
}
mask_path = '/Users/ahmadgill/Downloads/robotics project.v5-train-test-split_with_augmentation.png-mask-semantic/test/frame_0009_png.rf.99049d0355beec65f6cbbd6b983db658_mask.png'
mask = read_one_mask(mask_path)





def compute_speed_and_direction(
    mask,
    CLASS_NAMES,
    current_velocity=1.0,
    center_tolerance=10,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4,
):
    bottom_center = get_bottom_center_point(mask)

    distance_info = calculate_distance_and_side_from_bottom_center(
        mask=mask,
        bottom_center=bottom_center,
        class_names=CLASS_NAMES,
        center_tolerance=center_tolerance,
    )

    velocity_command = compute_velocity_from_distances(
        CLASS_NAMES=CLASS_NAMES,
        distance_info=distance_info,
        current_velocity=current_velocity,
        emergency_distance=emergency_distance,
        warning_distance=warning_distance,
        speed_breaker_distance=speed_breaker_distance,
        sidewalk_distance=sidewalk_distance,
        road_max_distance=road_max_distance,
        slow_factor=slow_factor,
    )

    return velocity_command
command = compute_speed_and_direction(
    mask=mask,
    CLASS_NAMES=CLASS_NAMES,
    current_velocity=1.0,
    center_tolerance=10,
    emergency_distance=90,
    warning_distance=180,
    speed_breaker_distance=160,
    sidewalk_distance=140,
    road_max_distance=40,
    slow_factor=0.4,
)
print(command)

{'linear_velocity': 0.4, 'direction': 'right', 'emergency_stop': False, 'reason': 'Sidewalk nearby - slow and steer away'}
